# Create scenario GDX from TYNDP 2020 data for GAMS

In [1]:
import pandas as pd
import numpy as np
import gdxtools as gt
import gams
import os
import sys
import datetime as dt
import calendar
import qgrid

In [2]:
#file path assignments
fn_additional = "../additional_data.xlsx"
fn_prices = '../source_data/TYNDP_2020/TYNDP_2020_Cost_Assumptions.xlsx'
fn_load_ts = "../parsed_data/TYNDP_2020/load_tyndp20.csv"
fn_res = "../parsed_data/TYNDP_2020/res_tyndp20.csv"
fn_cap = "../parsed_data/TYNDP_2020/cap_tyndp20.csv"
fn_ntc = "../parsed_data/TYNDP_2020/ntc_tyndp20.csv"
fn_chp = "../parsed_data/TYNDP_2020/chp_tyndp20.csv"

#dir_gdx = "../../"
# we export directly to the models data folder
dir_gdx = os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir) + os.sep + "model" + os.sep + "data"  + os.sep
dir_out = "../"

Weather year dependent data from normal model data

In [3]:
fn_ninja_tyndp = "../parsed_data/res_ninja_profiles_tyndp_years.csv"
fn_ror = "../parsed_data/hydro_ror_generation_hourly_ENTSO-E_adequacy.csv"
fn_reservoir_inflow = "../parsed_data/hydro_storage_inflows_weekly_ENTSO-E_adequacy.csv"

Scenario, runyear and weather year sets

In [4]:
scenario = ['DistributedEnergy', 'GlobalAmbition', 'NationalTrends']
runyear = [2025, 2030, 2040]
climateyear = [1982, 1984, 2007]

In [5]:
map_scenario = {
    'Distributed Energy':'DistributedEnergy', 
    'Global Ambition':'GlobalAmbition', 
    'National Trends':'NationalTrends'
}

Countries in model

In [6]:
#uncomment to include baltic states
df_countries= pd.read_excel(fn_additional, sheet_name='Countries_EU', index_col="Country")
countries = list(df_countries.index)

In [7]:
#uncomment to exclude baltic states
countries = ['AT', 'BE', 'BG', 'HR', 'CZ', 'DK', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LU', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE', 'CH', 'GB', 'NO']
df_countries = df_countries[df_countries.index.isin(countries)]

Trading partners for demand correction

In [8]:
df_countries_trade = pd.read_excel(fn_additional, sheet_name='Countries_EU_trade', index_col="Country")
countries_trade = list(df_countries_trade.index)

Technology sets

In [9]:
df_techs = pd.read_excel(fn_additional, sheet_name="Technologies")
storages = list(df_techs.Hydro.dropna().unique())
renewables = list(df_techs.Renewable.dropna().unique())
old_renewables = list(df_techs["Old Renewables"].dropna().unique())
all_renewables = renewables + old_renewables
conventionals = list(df_techs.Conventional.ffill().unique())
baseload = list(df_techs["Baseload"].dropna().unique())
fixed_feedin = list(df_techs["Fixed"].dropna().unique())
peakload = list(df_techs["peak"].dropna().unique())
technologies = storages + renewables + conventionals

## Prices

In [11]:
df_co2_price = pd.read_excel(fn_prices,sheet_name='CO2')
df_co2_price.scenario = df_co2_price.scenario.map(map_scenario)
df_co2_price = df_co2_price.set_index(['scenario','runyear'])
df_co2_price.head()

co2price
scenario          runyear          
NationalTrends    2030           28
                  2040           75
GlobalAmbition    2030           35
                  2040           80
DistributedEnergy 2030           53

## Hourly load

In [11]:
df_load_ts_in = pd.read_csv(fn_load_ts)
df_load_ts_in = df_load_ts_in[(df_load_ts_in['country'].isin(countries)                  
                    & df_load_ts_in['year'].isin(runyear)                 
                    )].rename(columns={'year':'runyear'})
df_load_ts_in.scenario = df_load_ts_in.scenario.map(map_scenario)
df_load_ts_in.head(1)

,scenario,runyear,country,time,value
0,NationalTrends,2025,AT,2025-01-01 00:00:00,7320.769525


In [12]:
df_load = df_load_ts_in.copy()
df_load['time'] = df_load['time'].str.replace("2025-","2017-")
df_load['time'] = df_load['time'].str.replace("2030-","2017-")
df_load['time'] = df_load['time'].str.replace("2040-","2017-")
df_load['time'] = pd.to_datetime(df_load['time'])
df_load = df_load.groupby(['scenario','runyear','country','time']).sum()
df_load.head()

value
scenario          runyear country time                            
DistributedEnergy 2030    AT      2017-01-01 00:00:00  9314.949567
                                  2017-01-01 01:00:00  9033.314552
                                  2017-01-01 02:00:00  8616.881420
                                  2017-01-01 03:00:00  8255.912582
                                  2017-01-01 04:00:00  7758.778216

## Capacities

In [13]:
df_cap_in = pd.read_csv(fn_cap)
df_cap = df_cap_in[(df_cap_in['country'].isin(countries)
                    & df_cap_in['technology'].isin(technologies)                 
                    & df_cap_in['runyear'].isin(runyear)
                    & df_cap_in['climateyear'].isin(climateyear)
                    )].copy()
df_cap.scenario = df_cap.scenario.map(map_scenario)
df_cap = df_cap.groupby(['scenario','runyear','climateyear','country','technology']).sum()
df_cap.head()

MW
scenario          runyear climateyear country technology             
DistributedEnergy 2030    1982        AT      Battery      534.675608
                                              Biomass      599.003723
                                              Gas         3415.699996
                                              Oil          168.406006
                                              Other        953.239990

# DSM capacities

In [14]:
df_dsm = df_cap_in[(df_cap_in['country'].isin(countries)
                    & df_cap_in['runyear'].isin(runyear)
                    & df_cap_in['climateyear'].isin(climateyear)
                    )].copy()
df_dsm = df_dsm[df_dsm['technology'] == 'DSR']
df_dsm.scenario = df_dsm.scenario.map(map_scenario)
df_dsm = df_dsm.groupby(['scenario','runyear','climateyear','country']).sum()
df_dsm.head()

MW
scenario          runyear climateyear country        
DistributedEnergy 2030    1982        BE       1800.0
                                      DE       5888.0
                                      ES       6000.0
                                      FI       4500.0
                                      FR       3400.0

## CHP generation

In [15]:
df_chp_in = pd.read_csv(fn_chp)
df_chp = df_chp_in[(df_chp_in['country'].isin(countries)
                    & df_chp_in['technology'].isin(technologies)
                    & df_chp_in['runyear'].isin(runyear)
                    & df_chp_in['climateyear'].isin(climateyear)
                    )]
df_chp.scenario = df_chp.scenario.map(map_scenario)
df_chp = df_chp.groupby(['scenario','runyear','climateyear','country','technology']).sum()
df_chp.head()

Value
scenario          runyear climateyear country technology              
DistributedEnergy 2030    1982        AT      Gas         4.733976e+06
                                      BE      Gas         5.433397e+06
                                      CH      Gas         3.610081e+06
                                      CZ      Gas         2.042677e+06
                                              Other       2.042677e+06

## Renewable generation

In [16]:
df_res_in = pd.read_csv(fn_res)
df_res = df_res_in[(df_res_in['country'].isin(countries)
                    & df_res_in['technology'].isin(technologies)
                    & df_res_in['runyear'].isin(runyear)
                    & df_res_in['climateyear'].isin(climateyear)
                    )]
df_res.scenario = df_res.scenario.map(map_scenario)
df_res = df_res.groupby(['scenario','runyear','climateyear','country','technology']).sum()
df_res.head()

MWh
scenario          runyear climateyear country technology               
DistributedEnergy 2030    1982        AT      Biomass      3.372631e+06
                                              RunOfRiver   3.448573e+07
                                              Solar        1.780784e+07
                                              WindOnshore  2.288679e+07
                                      BE      Biomass      8.461985e+05

In [17]:
df_res_ninja_in = pd.read_csv(fn_ninja_tyndp)
df_res_ninja = df_res_ninja_in[(df_res_ninja_in.country.isin(countries))
                     & (df_res_ninja_in.tech.isin(renewables)
                     )].copy().reset_index()
df_res_ninja['climateyear'] = pd.to_datetime(df_res_ninja['time']).dt.year
df_res_ninja = df_res_ninja[~((pd.to_datetime(df_res_ninja.time).dt.month == 2) & (pd.to_datetime(df_res_ninja.time).dt.day == 29))]
df_res_ninja['time'] = df_res_ninja['time'].str.replace("1982-","2017-")
df_res_ninja['time'] = df_res_ninja['time'].str.replace("1984-","2017-")
df_res_ninja['time'] = df_res_ninja['time'].str.replace("2007-","2017-")
df_res_ninja['time'] = pd.to_datetime(df_res_ninja['time'])
df_res_ninja.head()

,index,time,country,tech,capacity_factor,climateyear
0,736512,2017-01-01,AT,WindOffshore,0.0440,1982
1,736513,2017-01-01,BE,WindOffshore,0.0915,1982
2,736514,2017-01-01,BG,WindOffshore,0.1469,1982
3,736515,2017-01-01,CH,WindOffshore,0.0734,1982
4,736516,2017-01-01,CZ,WindOffshore,0.2292,1982


In [18]:
df_res_ninja_year = df_res_ninja.groupby(['climateyear','country','tech']).sum()
df_res_ninja_profile = df_res_ninja.merge(df_res_ninja_year,how='left',on=['climateyear','country','tech'])
df_res_ninja_profile['profile'] = df_res_ninja_profile['capacity_factor_x']/df_res_ninja_profile['capacity_factor_y']
df_res_ninja_profile = df_res_ninja_profile.set_index(['climateyear','country','tech','time'])[['profile']]
df_res_ninja_profile.head()

profile
climateyear country tech         time                
1982        AT      WindOffshore 2017-01-01  0.000022
            BE      WindOffshore 2017-01-01  0.000034
            BG      WindOffshore 2017-01-01  0.000076
            CH      WindOffshore 2017-01-01  0.000047
            CZ      WindOffshore 2017-01-01  0.000125

## RoR generation

In [19]:
df_ror_in = pd.read_csv(fn_ror)
df_ror_in = df_ror_in[df_ror_in.country.isin(countries)].copy()
df_ror_in = df_ror_in[~((pd.to_datetime(df_ror_in.date).dt.month == 2) & (pd.to_datetime(df_ror_in.date).dt.day == 29))]
df_ror_in['climateyear'] = pd.to_datetime(df_ror_in.date).dt.year
df_ror_in['date'] = df_ror_in['date'].str.replace("1982-","2017-")
df_ror_in['date'] = df_ror_in['date'].str.replace("1984-","2017-")
df_ror_in['date'] = df_ror_in['date'].str.replace("2007-","2017-")
df_ror_in.head()

,date,country,Run of River Hydro Generation in GWh per day,RoR generation MWh per hour,climateyear
1,2017-01-01,AT,81.135479,3380.644955,1982
3,2017-01-01,BE,1.919660,79.985843,1982
4,2017-01-01,BG,1.188977,49.540699,1982
5,2017-01-01,CH,35.233411,1468.058791,1982
6,2017-01-01,CZ,4.636210,193.175433,1982


In [20]:
#resample to hourly values by pivoting and then stacking
#and yes, there is probably a much smarter way for doing this...
df_ror_1982 = df_ror_in[df_ror_in['climateyear'] == 1982].copy()
df_ror_1984 = df_ror_in[df_ror_in['climateyear'] == 1984].copy()
df_ror_2007 = df_ror_in[df_ror_in['climateyear'] == 2007].copy()

df_ror_1982['date'] = pd.to_datetime(df_ror_1982['date'])
df_ror_1984['date'] = pd.to_datetime(df_ror_1984['date'])
df_ror_2007['date'] = pd.to_datetime(df_ror_2007['date'])

df_ror_1982_pivot = df_ror_1982.pivot_table(index='date',columns='country')[['RoR generation MWh per hour']].resample('H').fillna("pad").stack()
df_ror_1984_pivot = df_ror_1984.pivot_table(index='date',columns='country')[['RoR generation MWh per hour']].resample('H').fillna("pad").stack()
df_ror_2007_pivot = df_ror_2007.pivot_table(index='date',columns='country')[['RoR generation MWh per hour']].resample('H').fillna("pad").stack()

df_ror_1982_pivot['climateyear'] = 1982
df_ror_1984_pivot['climateyear'] = 1984
df_ror_2007_pivot['climateyear'] = 2007

df_ror_1982_pivot = df_ror_1982_pivot.reset_index().set_index(['climateyear','country','date'])
df_ror_1984_pivot = df_ror_1984_pivot.reset_index().set_index(['climateyear','country','date'])
df_ror_2007_pivot = df_ror_2007_pivot.reset_index().set_index(['climateyear','country','date'])

df_ror = pd.DataFrame()
df_ror = df_ror.append(df_ror_1982_pivot)
df_ror = df_ror.append(df_ror_1984_pivot)
df_ror = df_ror.append(df_ror_2007_pivot)

df_ror = df_ror.rename(columns={'RoR generation MWh per hour':'MWh'})

df_ror.head()

MWh
climateyear country date                   
1982        AT      2017-01-01  3380.644955
            BE      2017-01-01    79.985843
            BG      2017-01-01    49.540699
            CH      2017-01-01  1468.058791
            CZ      2017-01-01   193.175433

## NTC values

In [21]:
df_ntc_in = pd.read_csv(fn_ntc)
df_ntc_in = df_ntc_in[['from','to','scenario','runyear','climateyear','Export Capacity','Import Capacity']]
df_ntc_in = df_ntc_in[(df_ntc_in['from'].isin(countries)
                    & df_ntc_in['to'].isin(countries)
                    & df_ntc_in['runyear'].isin(runyear)
                    & df_ntc_in['climateyear'].isin(climateyear)
                    )]
df_ntc_in.scenario = df_ntc_in.scenario.map(map_scenario)
df_ntc_in = df_ntc_in.set_index(['scenario','runyear','climateyear','from','to'])
df_ntc_in.head(1)

,,,,,Export Capacity,Import Capacity
scenario,runyear,climateyear,from,to,,
DistributedEnergy,2030,1982,AT,CH,1200.0,1200.0


In [22]:
df_ntc_export = df_ntc_in[['Export Capacity']].rename(columns={'Export Capacity':'ntc'})
df_ntc_import = df_ntc_in[['Import Capacity']].rename(columns={'Import Capacity':'ntc'}).reset_index()
df_ntc_import['from_temp'] = df_ntc_import['from']
df_ntc_import['to_temp'] = df_ntc_import['to']
df_ntc_import['from'] = df_ntc_import['to_temp']
df_ntc_import['to'] = df_ntc_import['from_temp']
df_ntc_import = df_ntc_import.set_index(['scenario','runyear','climateyear','from','to'])[['ntc']]
df_ntc = df_ntc_export.append(df_ntc_import)
df_ntc.head()

ntc
scenario          runyear climateyear from to        
DistributedEnergy 2030    1982        AT   CH  1200.0
                          1984        AT   CH  1200.0
                          2007        AT   CH  1200.0
                  2040    1982        AT   CH  1200.0
                          1984        AT   CH  1200.0

## GDX export

In [23]:
def inject_static(gdx):
    """Injects static data into gdx file
    :param gdx: <gams.GamsDatabase> gdx to inject data
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # sets
    ## technology sets
    gt.add_set(scenario, "tyndpscenario", gdx, text="tyndp scenario", overwrite=overwrite)
    gt.add_set(runyear, "runyear", gdx, text="tyndp runyear", overwrite=overwrite)
    gt.add_set(climateyear, "climateyear", gdx, text="tyndp climateyear", overwrite=overwrite)
    
    gt.add_set(sorted(countries), "c", gdx, text="countries", overwrite=overwrite)
    gt.add_set(technologies, "tech", gdx, text="technologies", overwrite=overwrite)
         
    # capacities (now also including pumping)
    gt.add_parameter(df_cap.MW.to_dict(), "tyndp_cap", gdx,    
                  text="installed capacity [MW]", overwrite=overwrite)    
    
    # co2 prices 
    gt.add_parameter(df_co2_price.co2price.to_dict(), "tyndp_co2price", gdx,    
                  text="CO2 price [EUR/t]", overwrite=overwrite)    
    
    # DSM capacities
    gt.add_parameter(df_dsm.MW.to_dict(), "tyndp_cap_dsm", gdx,    
                  text="dsm capacity [MW]", overwrite=overwrite)    
      
        
    # yearly CHP generation by technology and country
    gt.add_parameter(df_chp.Value.to_dict(), "tyndp_chp", gdx,
                  text="yearly electricity generation from chp plants by country and technology [MWh]",
                  overwrite = overwrite)
    
    # ntc between countries
    gt.add_parameter(df_ntc.ntc.to_dict(), "tyndp_ntc", gdx,
                  text="ntc from country to country [Mh]",
                  overwrite = overwrite)    
    
    # yearly res generation
    gt.add_parameter(df_res.MWh.to_dict(), "tyndp_gen_annual", gdx,
                  text="annual renewable electricity generation by country and technology [MWh]",
                  overwrite = overwrite)    
    

In [24]:
def inject_dynamic(gdx, df_period):
    """ Injects time dependent parameters into gdx given period selection
    :param gdx: <gams.GamsDatabase> gdx to inject data
    :param df_periods: <pd.DataFrame> with mapping from dates to periods and lenght of respective period
    """
    # overwrite settings
    overwrite = True
    
    # extract mapping dates to periods
    map_periods = df_periods.period.to_dict()
    periods = map_periods.values()
    
    # sets
    gt.add_set(df_periods.period, "t", gdx, text="Periods", overwrite=overwrite)
            
    # load parameter 
    df_ = df_load.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["scenario","runyear","country", "period"])
    gt.add_parameter(df_.value.to_dict(), "tyndp_demand", gdx,
    text="tyndp demand per period [MWh]", overwrite=overwrite)  
    
    # renewable supply
    df_ = df_res_ninja_profile.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(['climateyear','country','tech',"period"])
    gt.add_parameter(df_.profile.to_dict(), "tyndp_renS", gdx,
    text="renewable ninja generation share for tyndp years [%]", overwrite=overwrite)  
    
    # ror generation
    df_ = df_ror.reset_index()
    df_["period"] = df_.date.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(['climateyear','country',"period"])
    gt.add_parameter(df_.MWh.to_dict(), "tyndp_ror", gdx,
    text="ror generation for tyndp years", overwrite=overwrite)  

Create a mapping from dates to time period in the model:

In [25]:
df_periods = pd.Series({d: "t%04d" % (i+1) for i, d in enumerate(df_load.reset_index().time.unique())}).to_frame("period")
df_periods["periodLength"] = 1

Create gdx and workspace

In [26]:
dir_gms = os.getcwd()
ws = gams.GamsWorkspace(dir_gms)
gdx_all = ws.add_database()

Inject to gdx file

In [27]:
inject_static(gdx_all)

In [28]:
inject_dynamic(gdx_all, df_periods)

Save gdx

In [29]:
fn_out = dir_gdx + "scenario_data_tyndp.gdx"
gdx_all.export(fn_out)